In [ ]:
import requests
import pandas as pd 

docs_url = 'https://github.com/alexeygrigorev/llm-rag-workshop/raw/main/notebooks/documents.json'
docs_response = requests.get(docs_url)
documents_raw = docs_response.json()

documents = []

for doc in documents_raw:
    course_name = doc['course']
    for doc in doc['documents']:
        doc['course'] = course_name
        documents.append(doc)


In [ ]:
import minsearch

index = minsearch.Index(
    text_fields=['question','text','section'],
    keyword_fields=['course'],
)
index.fit(documents)

In [ ]:
from openai import OpenAI
import os


In [ ]:
client = OpenAI(
    base_url='https://openrouter.ai/api/v1',
    api_key=os.getenv("OPENROUTERAI_API_KEY")  # OpenRouter base URL
)


In [ ]:
def search(query):
    boost = {'question': 3.0, 'section': 1.0}

    results = index.search(query, filter_dict={'course':'data-engineering-zoomcamp'}, boost_dict=boost, num_results=5)
    return results

def build_prompt(query, search_results):
    prompt_template = """
You're a course teaching assistant. Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.

QUESTION: {question}

CONTEXT:
{context}
""".strip()

    context_template = """
Q: {question}
A: {text}
""".strip()
    
    context = ""
    for result in search_results:
        context += context_template.format(question=result['question'], text=result['text']) + "\n"
    
    prompt = prompt_template.format(question=query, context=context)
    return prompt

def llm(prompt):
    response = client.chat.completions.create(
      model="qwen/qwen3-4b:free",
      messages=[
        {
          "role": "user",
          "content": prompt,
        },
      ]
    )

    return response.choices[0].message.content

def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt)
    return answer

query = 'how do I enroll the course?'
answer = rag(query)



In [ ]:
answer